# 06 - Analisis biaya retrieval dan ekspor hasil

Tiga hal: mengukur biaya indeks FAISS yang menjadi satu-satunya biaya tambahan
RM-c, menggabungkan riwayat run bila ada lebih dari satu folder kampanye, dan
mengekspor seluruh artefak ke satu berkas Excel untuk penulisan Bab 4.

In [1]:
import pandas as pd

from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.output_dir / "tuning_lite"
runner = CampaignRunner(
    out_dir=OUT_DIR, model_name="indobenchmark/indobert-lite-base-p2"
)
features = runner.features
print("fitur beku:", {k: v.shape for k, v in features.embeddings.items()})

c:\Penelitian\IndoBERT-with-RAC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-08 13:14:35,499 | INFO     | src.services.data | Data dimuat: train=6588 val=1402 test=1405 | device=cuda | encoder=indobenchmark/indobert-lite-base-p2


2026-09-08 13:14:38,648 | WARNING  | src.models.comment_dataset | AutoTokenizer untuk 'indobenchmark/indobert-lite-base-p2' menghasilkan vocab mencurigakan kecil (5); fallback ke BertTokenizer (WordPiece)
2026-09-08 13:14:39,928 | INFO     | src.services.features | Fitur beku dimuat dari cache C:\Penelitian\IndoBERT-with-RAC\outputs\tuning_lite\features\indobenchmark__indobert-lite-base-p2
fitur beku: {'train': (6588, 768), 'val': (1402, 768), 'test': (1405, 768)}


## 1. Biaya indeks FAISS

RM-c mengklaim nol waktu latih. Klaim itu baru jujur bila biaya retrieval ikut
diukur, dan biayanya terbagi dua: pembangunan indeks sekali dari embedding train,
serta penelusuran pada setiap inferensi yang tumbuh mengikuti k.

In [2]:
from src.services.faiss_benchmark import FaissBenchmark

benchmark = FaissBenchmark(
    features.embeddings["train"], features.labels["train"], repeats=5
)
hasil = benchmark.run(
    features.embeddings["test"],
    k_values=(1, 3, 5, 10, 20, 50),
    out_dir=OUT_DIR / "faiss_index",
)

print(pd.Series(hasil.build).to_string())
pd.DataFrame(hasil.search)

2026-09-08 13:14:40,222 | INFO     | src.services.faiss_benchmark | Bangun indeks: 6588 vektor x 768 dim dalam 33.27 ms (19.30 MB)
2026-09-08 13:14:41,706 | INFO     | src.services.faiss_benchmark | Telusur k=1: 269.453 ms untuk 1405 query (191.78 us/query)
2026-09-08 13:14:43,318 | INFO     | src.services.faiss_benchmark | Telusur k=3: 321.543 ms untuk 1405 query (228.86 us/query)
2026-09-08 13:14:44,572 | INFO     | src.services.faiss_benchmark | Telusur k=5: 250.025 ms untuk 1405 query (177.95 us/query)
2026-09-08 13:14:45,881 | INFO     | src.services.faiss_benchmark | Telusur k=10: 261.217 ms untuk 1405 query (185.92 us/query)
2026-09-08 13:14:47,252 | INFO     | src.services.faiss_benchmark | Telusur k=20: 273.507 ms untuk 1405 query (194.67 us/query)
2026-09-08 13:14:48,635 | INFO     | src.services.faiss_benchmark | Telusur k=50: 275.851 ms untuk 1405 query (196.34 us/query)
2026-09-08 13:14:48,676 | INFO     | src.services.faiss_benchmark | Hasil benchmark FAISS ditulis ke C:\

,k,n_queries,search_time_ms_batch,search_time_us_per_query
0,1,1405,269.4532,191.7817
1,3,1405,321.5433,228.8564
2,5,1405,250.0253,177.9540
3,10,1405,261.2166,185.9193
4,20,1405,273.5068,194.6668
5,50,1405,275.8514,196.3355


Indeks bertipe flat/exact, jadi penelusuran adalah brute force atas seluruh
vektor train. Untuk ukuran data ini biayanya sepele dan hasilnya deterministik,
yang jauh lebih penting untuk penelitian daripada penghematan waktu dari indeks
aproksimasi.

In [3]:
import matplotlib.pyplot as plt

search = pd.DataFrame(hasil.search)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(search["k"], search["search_time_us_per_query"], marker="o")
ax.set_xlabel("k (jumlah tetangga)")
ax.set_ylabel("mikrodetik per query")
ax.set_title("Biaya penelusuran indeks FAISS")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

C:\Users\Arya\AppData\Local\Temp\ipykernel_47656\4266193051.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Gabungkan riwayat lintas folder kampanye

Hanya perlu bila ada lebih dari satu folder keluaran, misalnya kampanye utama
ditambah eksplorasi dengan encoder berbeda.

In [4]:
from src.services.aggregation import RunMerger

SUMBER = {"tuning": OUT_DIR}

if len(SUMBER) > 1:
    ditulis = RunMerger(SUMBER).merge_all(OUT_DIR / "combined")
    for kunci, path in ditulis.items():
        print(f"  {kunci}: {path}")
else:
    print("hanya satu folder kampanye; penggabungan dilewati")

hanya satu folder kampanye; penggabungan dilewati


Saat membaca hasil gabungan, kunci barisnya adalah pasangan (`source`, `run_id`)
karena tiap folder memulai penomoran dari 1. Kolom waktu, memori, dan latency
tidak boleh dibandingkan lintas `source`.

## 3. Ekspor ke Excel

In [5]:
from src.services.workbook import WorkbookBuilder

builder = WorkbookBuilder(OUT_DIR)
path = builder.build(settings.data_dir.parent / "HASIL_lite.xlsx")
print(f"{path} ({path.stat().st_size / 1024:.0f} KB)")

for nama, frame in builder.sheets().items():
    print(f"  {nama:28s}: {len(frame):4,} baris x {len(frame.columns)} kolom")

2026-09-08 13:14:52,114 | INFO     | src.services.workbook | Workbook ditulis ke C:\Penelitian\IndoBERT-with-RAC\HASIL.xlsx (14 sheet)
C:\Penelitian\IndoBERT-with-RAC\HASIL.xlsx (60 KB)
  Ringkasan                   :   27 baris x 2 kolom
  Run RMA                     :   28 baris x 27 kolom
  Kurva RMA                   :  148 baris x 8 kolom
  Run RMB                     :   27 baris x 29 kolom
  Kurva RMB                   :  345 baris x 8 kolom
  Run RMC                     :   67 baris x 21 kolom
  Perbandingan Final          :    3 baris x 12 kolom
  Kriteria Sukses             :    2 baris x 11 kolom
  Benchmark Inferensi         :    3 baris x 3 kolom
  Pivot rma_grid_pivot_batch16:    4 baris x 4 kolom
  Pivot rma_grid_pivot_batch32:    4 baris x 4 kolom
  Pivot rmb_grid_pivot_hidden_dim:    1 baris x 2 kolom
  Pivot rmc_grid_pivot_weightings:   11 baris x 7 kolom
  Pivot rmc_grid_pivot_weightingu:    1 baris x 2 kolom


## 4. Regenerasi seluruh figur

In [6]:
from src.services.reporting import FigureReporter

reporter = FigureReporter(OUT_DIR)
for skenario in ("rma", "rmb", "rmc"):
    dibuat = reporter.refresh_scenario(skenario)
    print(f"{skenario}: {len(dibuat)} artefak")

reporter.final_inference_bar_chart()
reporter.write_summary()
print(f"\ntotal figur: {len(list((OUT_DIR / 'figures').glob('*.png')))}")

rma: 6 artefak
rmb: 12 artefak
rmc: 6 artefak

total figur: 81


## Ringkasan

Bahan Bab 4 lengkap: `HASIL.xlsx` di root, tabel metrik di
`outputs/tuning/metrics/`, figur di `outputs/tuning/figures/`, dan biaya
retrieval di `outputs/faiss_index/`.